# Module 2 — Python for AI Engineering
Hands-on: build a production-shaped async LLM client with validation, concurrency, timeouts, retries, and cost accounting.


In [ ]:
import asyncio, time
from dataclasses import dataclass

@dataclass
class Response:
    text: str
    tokens: int

async def fake_llm(prompt, delay=.05, fail=False):
    await asyncio.sleep(delay)
    if fail: raise TimeoutError('provider timeout')
    return Response('answer: '+prompt[:30], len(prompt.split())+8)


In [ ]:
async def ask(prompt, retries=2):
    for attempt in range(retries+1):
        try: return await fake_llm(prompt)
        except TimeoutError:
            if attempt == retries: raise
            await asyncio.sleep(0.05 * 2**attempt)

print(asyncio.run(ask('Explain embeddings simply')))


## Lab 1 — Concurrency
Run 20 requests sequentially and concurrently. Measure wall-clock latency. Then add a semaphore and compare concurrency 2, 5, and 10.


In [ ]:
async def batch(n=20, limit=None):
    sem = asyncio.Semaphore(limit) if limit else None
    async def one(i):
        if sem:
            async with sem: return await ask(f'question {i}')
        return await ask(f'question {i}')
    t=time.perf_counter(); out=await asyncio.gather(*(one(i) for i in range(n))); return time.perf_counter()-t,out
for limit in [1,2,5,10]: print(limit, asyncio.run(batch(20,limit))[0])


## Lab 2 — Failure engineering
Inject timeouts, malformed responses, cancellation, and retry storms. Decide which errors are retryable. Add a hard retry budget and exponential backoff.

## Lab 3 — Production contract
Add request IDs, structured errors, token/cost estimates, timeout deadlines, and a provider-neutral interface.

**Exercises:** async context manager, cancellation-safe cleanup, circuit breaker, rate limiter, idempotency key, batch API, streaming simulation, property tests.


### Mastery gate
You can explain async I/O vs threads, demonstrate bounded concurrency, classify retryable errors, and show that a failure cannot silently become an infinite retry loop.